# 03 · Modelado, evaluación y exportación a Power BI
**Tesis:** Modelo de ML basado en series temporales para la predicción de eventos climáticos extremos en la región de Puno con datos del SENAMHI (UNI – Maestría en Ciencia de la Computación). **Autor:** Antonio Arthur Quispe Javier. 

Entrena XGBoost, LSTM y TFT (simplificado, PyTorch) para predecir precipitación y temperatura media a 1 día y detectar lluvia intensa (≥ P95 de días húmedos) y día frío extremo (≤ P5). División temporal: entrenamiento 1964–2010, validación 2011–2017, prueba 2018–2026.

In [ ]:
# Montar Google Drive y definir rutas (ajuste DATA a la carpeta con los .xlsx del SENAMHI)
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/seminario_tesis_II/data'     # <- archivos ayaviri.xlsx, ILAVE.xlsx, JULI.xlsx, ...
OUT  = '/content/drive/MyDrive/seminario_tesis_II/out'
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
%%writefile common.py
"""Funciones comunes: carga, limpieza, eventos, ventanas, métricas."""
import glob, os, numpy as np, pandas as pd

ESTACIONES = ["PUNO","JULI","ILAVE","AYAVIRI","MAZOCRUZ","PAMPAHUTA","CHUQUIBAMBILLA","MACUSANI"]
MODELADAS  = ["PUNO","JULI","ILAVE","AYAVIRI","MAZOCRUZ","PAMPAHUTA","CHUQUIBAMBILLA"]  # Macusani excluida (cobertura TM 16 %)
TRAIN_END, VAL_END = "2010-12-31", "2017-12-31"
WIN = 30

def cargar_estacion(path):
    d = pd.read_excel(path, header=1)
    d.columns = [str(c).strip() for c in d.columns]
    fecha = pd.to_datetime(dict(year=d["Año"], month=d["Mes"], day=d["V Dia"]), errors="coerce")
    out = pd.DataFrame({"fecha": fecha, "HR": d["HR101"], "PT": d["PT101"], "TM": d["TM101"]}).dropna(subset=["fecha"])
    for c in ["HR","PT","TM"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out["PT_888"] = (out["PT"] == -888)
    for c in ["HR","PT","TM"]:
        out.loc[out[c] == -999, c] = np.nan              # -999 = sin dato (SENAMHI)
    out.loc[out["PT"] == -888, "PT"] = 0.0               # -888 = traza (precipitación inapreciable) -> 0 mm
    for c in ["HR","TM"]:
        out.loc[out[c] == -888, c] = np.nan
    out = out.drop_duplicates("fecha").set_index("fecha").sort_index()
    return out

def cargar_todo(carpeta):
    S = {}
    for f in glob.glob(os.path.join(carpeta, "*.xlsx")):
        name = os.path.splitext(os.path.basename(f))[0].upper()
        if name in ESTACIONES:
            d = cargar_estacion(f)
            idx = pd.date_range("1964-01-01", "2026-08-31", freq="D")
            S[name] = d.reindex(idx)
            S[name]["PT_888"] = S[name]["PT_888"].fillna(False).astype(bool)
    return S

def rangos_fisicos(S):
    """Controles de calidad: valores fuera de rango físico -> NaN. Devuelve conteo."""
    rep = []
    for k, d in S.items():
        n0 = d[["HR","PT","TM"]].notna().sum()
        d.loc[(d.PT < 0) | (d.PT > 300), "PT"] = np.nan
        d.loc[(d.HR < 0) | (d.HR > 100), "HR"] = np.nan
        d.loc[(d.TM < -30) | (d.TM > 40), "TM"] = np.nan
        n1 = d[["HR","PT","TM"]].notna().sum()
        rep.append((k, int((n0-n1).sum())))
    return rep

def tabla_cobertura(S):
    rows = []
    for k, d in S.items():
        r = dict(Estacion=k, Inicio=d.index.min().date(), Fin=d.index.max().date(), Dias=len(d))
        for c in ["PT","TM","HR"]:
            r[f"%{c}"] = round(100*d[c].notna().mean(), 1)
        r["PT_-888"] = int(d.PT_888.sum())
        rows.append(r)
    return pd.DataFrame(rows)

def umbrales(S):
    """Umbrales de evento por estación con el periodo de entrenamiento únicamente."""
    U = {}
    for k, d in S.items():
        tr = d.loc[:TRAIN_END]
        pt = tr.PT.dropna()
        U[k] = dict(p95_lluvia=float(pt[pt >= 1.0].quantile(0.95)), p5_tm=float(tr.TM.dropna().quantile(0.05)))
    return U

def panel_supervisado(S, U, estaciones, h=1):
    """Devuelve dict con arrays por estación listos para ventanas deslizantes."""
    P = {}
    for k in estaciones:
        d = S[k].copy()
        d["y_PT"] = d.PT.shift(-h); d["y_TM"] = d.TM.shift(-h)
        d["e_lluvia"] = (d.y_PT >= U[k]["p95_lluvia"]).astype(float).where(d.y_PT.notna())
        d["e_frio"] = (d.y_TM <= U[k]["p5_tm"]).astype(float).where(d.y_TM.notna())
        P[k] = d
    return P

def clima_doy(S, estaciones):
    """Climatología por día del año (media suavizada) usando solo entrenamiento."""
    C = {}
    for k in estaciones:
        tr = S[k].loc[:TRAIN_END]
        g = tr[["PT","TM","HR"]].groupby(tr.index.dayofyear).mean().reindex(range(1, 367)).interpolate(limit_direction="both")
        g = pd.concat([g.iloc[-15:], g, g.iloc[:15]]).rolling(31, center=True).mean().iloc[15:-15]
        g.index = range(1, 367); C[k] = g
    return C

def features_tabulares(P, S, C):
    """Rezagos y estadísticos móviles para XGBoost. Retorna DataFrame concatenado."""
    frames = []
    for si, (k, d) in enumerate(P.items()):
        f = pd.DataFrame(index=d.index)
        for v in ["PT","TM","HR"]:
            for L in [0,1,2,3,4,5,6]:
                f[f"{v}_l{L}"] = d[v].shift(L)
        f["PT_s3"]  = d.PT.rolling(3, min_periods=2).sum()
        f["PT_s7"]  = d.PT.rolling(7, min_periods=4).sum()
        f["PT_s15"] = d.PT.rolling(15, min_periods=8).sum()
        f["PT_s30"] = d.PT.rolling(30, min_periods=15).sum()
        f["TM_m7"]  = d.TM.rolling(7, min_periods=4).mean()
        f["TM_m30"] = d.TM.rolling(30, min_periods=15).mean()
        f["HR_m7"]  = d.HR.rolling(7, min_periods=4).mean()
        f["HR_m30"] = d.HR.rolling(30, min_periods=15).mean()
        f["TM_s7"]  = d.TM.rolling(7, min_periods=4).std()
        wet = (d.PT >= 1.0).astype(float).where(d.PT.notna())
        f["dias_hum30"] = wet.rolling(30, min_periods=15).sum()
        # días desde la última lluvia >= 1 mm (tope 60)
        last = pd.Series(np.where(wet == 1, np.arange(len(d)), np.nan), index=d.index).ffill()
        f["dias_sin_lluvia"] = (np.arange(len(d)) - last).clip(upper=60)
        doy = d.index.dayofyear.values
        f["sin_doy"] = np.sin(2*np.pi*(doy+1)/365.25); f["cos_doy"] = np.cos(2*np.pi*(doy+1)/365.25)
        f["TM_anom"] = d.TM - C[k]["TM"].reindex(doy).values
        f["estacion"] = si
        for t in ["y_PT","y_TM","e_lluvia","e_frio"]:
            f[t] = d[t]
        f["clima_PT"] = C[k]["PT"].reindex(((doy) % 366) + 1).values   # climatología del día objetivo (aprox.)
        f["clima_TM"] = C[k]["TM"].reindex(((doy) % 366) + 1).values
        f["nombre"] = k
        frames.append(f)
    return pd.concat(frames)

def split_de(idx):
    s = np.where(idx <= pd.Timestamp(TRAIN_END), 0, np.where(idx <= pd.Timestamp(VAL_END), 1, 2))
    return s

def metricas_reg(y, p):
    m = ~(np.isnan(y) | np.isnan(p)); y, p = y[m], p[m]
    rmse = float(np.sqrt(np.mean((y-p)**2))); mae = float(np.mean(np.abs(y-p)))
    r2 = float(1 - np.sum((y-p)**2)/np.sum((y-y.mean())**2))
    return dict(RMSE=rmse, MAE=mae, R2=r2, n=int(m.sum()))

def prf(y, pred):
    y = y.astype(int); pred = pred.astype(int)
    tp = int(((y==1)&(pred==1)).sum()); fp = int(((y==0)&(pred==1)).sum()); fn = int(((y==1)&(pred==0)).sum())
    pr = tp/(tp+fp) if tp+fp else 0.0; rc = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*pr*rc/(pr+rc) if pr+rc else 0.0
    return dict(Precision=pr, Recall=rc, F1=f1, TP=tp, FP=fp, FN=fn)

def mejor_umbral(y, score, grid=None):
    """Umbral de decisión que maximiza F1 en validación."""
    m = ~(np.isnan(y) | np.isnan(score)); y, s = y[m], score[m]
    grid = np.unique(np.quantile(s, np.linspace(0.5, 0.999, 200))) if grid is None else grid
    best, bt = -1, grid[0]
    for t in grid:
        f = prf(y, s >= t)["F1"]
        if f > best: best, bt = f, t
    return float(bt)

def auc_pr(y, s):
    m = ~(np.isnan(y) | np.isnan(s)); y, s = y[m], s[m]
    o = np.argsort(-s); y = y[o]
    tp = np.cumsum(y); fp = np.cumsum(1-y)
    prec = tp/np.maximum(tp+fp, 1); rec = tp/max(y.sum(), 1)
    return float(np.sum(np.diff(np.concatenate([[0.0], rec]))*prec))


In [ ]:
%%writefile run_xgb.py
import sys, json, pickle, numpy as np, pandas as pd, xgboost as xgb, time
from common import *
DATA = sys.argv[1]; OUT = sys.argv[2]
S = cargar_todo(DATA); rangos_fisicos(S)
U = umbrales(S); C = clima_doy(S, MODELADAS)
P = panel_supervisado(S, U, MODELADAS)
F = features_tabulares(P, S, C)
F["split"] = split_de(F.index)
feat = [c for c in F.columns if c not in ["y_PT","y_TM","e_lluvia","e_frio","split","nombre","clima_PT","clima_TM"]]
res = {}; preds = F[["nombre","split","y_PT","y_TM","e_lluvia","e_frio"]].copy()
t0 = time.time()
for tgt, ev in [("y_PT","e_lluvia"),("y_TM","e_frio")]:
    m = F[tgt].notna()
    tr = F[m & (F.split==0)]; va = F[m & (F.split==1)]; te = F[m & (F.split==2)]
    reg = xgb.XGBRegressor(n_estimators=600, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                           min_child_weight=5, early_stopping_rounds=40, tree_method="hist", n_jobs=2, random_state=42)
    reg.fit(tr[feat], tr[tgt], eval_set=[(va[feat], va[tgt])], verbose=False)
    preds.loc[F.index.isin(F.index) , f"xgb_{tgt}"] = np.nan
    preds[f"xgb_{tgt}"] = reg.predict(F[feat])
    me = F[ev].notna()
    tr = F[me & (F.split==0)]; va = F[me & (F.split==1)]
    spw = float((1-tr[ev].mean())/tr[ev].mean())
    clf = xgb.XGBClassifier(n_estimators=600, learning_rate=0.03, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                            min_child_weight=5, scale_pos_weight=min(spw, 8), early_stopping_rounds=40, eval_metric="aucpr",
                            tree_method="hist", n_jobs=2, random_state=42)
    clf.fit(tr[feat], tr[ev], eval_set=[(va[feat], va[ev])], verbose=False)
    preds[f"xgb_p_{ev}"] = clf.predict_proba(F[feat])[:, 1]
    res[tgt] = dict(best_iter_reg=int(reg.best_iteration), best_iter_clf=int(clf.best_iteration))
    imp = pd.Series(reg.feature_importances_, index=feat).sort_values(ascending=False)
    imp.to_csv(f"{OUT}/xgb_importancia_{tgt}.csv")
res["tiempo_s"] = time.time()-t0; res["n_features"] = len(feat); res["xgboost"] = xgb.__version__
json.dump(res, open(f"{OUT}/xgb_info.json","w"), indent=1)
preds.to_pickle(f"{OUT}/preds_xgb.pkl")
pickle.dump(dict(U=U), open(f"{OUT}/umbrales.pkl","wb"))
F[["nombre","split","y_PT","y_TM","e_lluvia","e_frio","clima_PT","clima_TM","PT_l0","TM_l0"]].to_pickle(f"{OUT}/base.pkl")
tabla_cobertura(S).to_csv(f"{OUT}/cobertura.csv", index=False)
print(res)


In [ ]:
%%writefile dl.py
"""LSTM y TFT simplificado (PyTorch) multitarea: PT(t+1), TM(t+1), evento lluvia, evento frío."""
import sys, json, time, numpy as np, pandas as pd, torch, torch.nn as nn
from common import *
torch.set_num_threads(2)

def construir(S, U, C):
    P = panel_supervisado(S, U, MODELADAS)
    # normalización con entrenamiento
    st = {}
    X, Y, E, ST, ID, DT, SP = [], [], [], [], [], [], []
    for si, k in enumerate(MODELADAS):
        d = P[k]; tr = d.loc[:TRAIN_END]
        mu = dict(PT=np.log1p(tr.PT.dropna()).mean(), TM=tr.TM.mean(), HR=tr.HR.mean())
        sd = dict(PT=np.log1p(tr.PT.dropna()).std(), TM=tr.TM.std(), HR=tr.HR.std())
        pt = (np.log1p(d.PT) - mu["PT"])/sd["PT"]; tm = (d.TM-mu["TM"])/sd["TM"]; hr = (d.HR-mu["HR"])/sd["HR"]
        doy = d.index.dayofyear.values
        arr = np.stack([pt, tm, hr, pt.isna(), tm.isna(), hr.isna(),
                        np.sin(2*np.pi*doy/365.25), np.cos(2*np.pi*doy/365.25)], 1).astype("float32")
        arr[:, :3] = np.nan_to_num(arr[:, :3], nan=0.0)
        n = len(d)
        idx = np.arange(WIN-1, n)
        win = np.stack([arr[i-WIN+1:i+1] for i in idx])
        y = d[["y_PT","y_TM"]].values[idx]; e = d[["e_lluvia","e_frio"]].values[idx]
        ok = ~np.isnan(y).all(1)
        X.append(win[ok]); Y.append(y[ok]); E.append(e[ok]); ST.append(np.full(ok.sum(), si)); DT.append(d.index[idx][ok])
        st[k] = dict(mu=mu, sd=sd)
    X = np.concatenate(X); Y = np.concatenate(Y); E = np.concatenate(E); ST = np.concatenate(ST); DT = np.concatenate(DT)
    SP = split_de(pd.DatetimeIndex(DT))
    return X, Y, E, ST, DT, SP, st

class LSTMNet(nn.Module):
    def __init__(s, nin=8, h=64, ns=7):
        super().__init__()
        s.emb = nn.Embedding(ns, 8); s.lstm = nn.LSTM(nin, h, batch_first=True, num_layers=1)
        s.head = nn.Sequential(nn.Linear(h+8, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 4))
    def forward(s, x, st):
        o, (hn, _) = s.lstm(x); return s.head(torch.cat([hn[-1], s.emb(st)], 1))

class GLU_GRN(nn.Module):
    """Gated Residual Network (Lim et al., 2021)."""
    def __init__(s, d, dout=None, dctx=None, drop=0.1):
        super().__init__(); dout = dout or d
        s.l1 = nn.Linear(d, d); s.ctx = nn.Linear(dctx, d, bias=False) if dctx else None
        s.l2 = nn.Linear(d, dout*2); s.drop = nn.Dropout(drop); s.norm = nn.LayerNorm(dout)
        s.skip = nn.Linear(d, dout) if d != dout else nn.Identity(); s.dout = dout
    def forward(s, x, c=None):
        h = s.l1(x) + (s.ctx(c) if (c is not None and s.ctx) else 0)
        h = s.l2(s.drop(nn.functional.elu(h))); a, b = h.chunk(2, -1)
        return s.norm(s.skip(x) + a*torch.sigmoid(b))

class TFT(nn.Module):
    """TFT simplificado: selección de variables (GRN+softmax), codificación estática por estación,
    LSTM codificador, auto-atención multi-cabeza interpretable y compuertas residuales."""
    def __init__(s, nvar=6, dm=48, ns=7, heads=4):
        super().__init__()
        s.nvar = nvar; s.dm = dm
        s.emb = nn.Embedding(ns, dm); s.static_ctx = GLU_GRN(dm)
        s.var_proj = nn.ModuleList([nn.Linear(1, dm) for _ in range(nvar)])
        s.var_grn = nn.ModuleList([GLU_GRN(dm) for _ in range(nvar)])
        s.sel = GLU_GRN(nvar*dm, nvar, dctx=dm)
        s.lstm = nn.LSTM(dm, dm, batch_first=True)
        s.gate1 = nn.Linear(dm, dm*2); s.n1 = nn.LayerNorm(dm)
        s.enh = GLU_GRN(dm, dctx=dm)
        s.att = nn.MultiheadAttention(dm, heads, batch_first=True)
        s.gate2 = nn.Linear(dm, dm*2); s.n2 = nn.LayerNorm(dm)
        s.pos = GLU_GRN(dm); s.out = nn.Linear(dm, 4)
        s.last_weights = None
    def forward(s, x, st):
        c = s.static_ctx(s.emb(st))                                  # contexto estático
        # variables de entrada continuas: PT, TM, HR, sin, cos, (3 máscaras compactadas en 1 promedio)
        v = torch.cat([x[:, :, :3], x[:, :, 6:8], x[:, :, 3:6].mean(2, keepdim=True)], 2)
        tok = torch.stack([g(p(v[:, :, i:i+1])) for i, (p, g) in enumerate(zip(s.var_proj, s.var_grn))], 2)  # B,T,V,dm
        w = torch.softmax(s.sel(tok.flatten(2), c[:, None, :].expand(-1, x.size(1), -1)), -1)   # B,T,V
        s.last_weights = w.detach()
        z = (tok*w.unsqueeze(-1)).sum(2)
        o, _ = s.lstm(z)
        a, b = s.gate1(o).chunk(2, -1); o = s.n1(z + a*torch.sigmoid(b))
        o = s.enh(o, c[:, None, :].expand(-1, o.size(1), -1))
        T = o.size(1); mask = torch.triu(torch.ones(T, T, dtype=torch.bool), 1)
        at, _ = s.att(o, o, o, attn_mask=mask)
        a, b = s.gate2(at).chunk(2, -1); o2 = s.n2(o + a*torch.sigmoid(b))
        return s.out(s.pos(o2)[:, -1])

def entrenar(model_fn, X, Y, E, ST, SP, ym, ys, seed=42, epochs=14, bs=512, lr=2e-3, pw=(4.0, 4.0), log=None):
    torch.manual_seed(seed); np.random.seed(seed)
    m = model_fn(); opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
    Xt = torch.tensor(X); St = torch.tensor(ST)
    Yn = torch.tensor(((Y-ym)/ys).astype("float32")); En = torch.tensor(np.nan_to_num(E).astype("float32"))
    Mr = torch.tensor(~np.isnan(Y)); Me = torch.tensor(~np.isnan(E))
    tri = np.where(SP==0)[0]; vai = np.where(SP==1)[0]
    bce = nn.BCEWithLogitsLoss(reduction="none", pos_weight=torch.tensor(list(pw)))
    def loss_fn(o, i):
        lr_ = ((o[:, :2]-torch.nan_to_num(Yn[i]))**2*Mr[i]).sum()/Mr[i].sum()
        le = (bce(o[:, 2:], En[i])*Me[i]).sum()/Me[i].sum()
        return lr_ + 0.5*le
    best, bstate, bad = 1e9, None, 0
    for ep in range(epochs):
        m.train(); np.random.shuffle(tri); t0 = time.time(); tl = 0
        for j in range(0, len(tri), bs):
            i = torch.tensor(tri[j:j+bs]); opt.zero_grad(); l = loss_fn(m(Xt[i], St[i]), i); l.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); tl += l.item()*len(i)
        m.eval(); vl = 0
        with torch.no_grad():
            for j in range(0, len(vai), 2048):
                i = torch.tensor(vai[j:j+2048]); vl += loss_fn(m(Xt[i], St[i]), i).item()*len(i)
        vl /= len(vai); sched.step(vl)
        msg = f"ep{ep+1} train {tl/len(tri):.4f} val {vl:.4f} ({time.time()-t0:.0f}s)"; print(msg, flush=True)
        if log is not None: log.append(msg)
        if vl < best - 1e-4: best, bstate, bad = vl, {k: v.clone() for k, v in m.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= 4: break
    m.load_state_dict(bstate); m.eval(); return m

def predecir(m, X, ST, ym, ys):
    out = []
    with torch.no_grad():
        for j in range(0, len(X), 4096):
            out.append(m(torch.tensor(X[j:j+4096]), torch.tensor(ST[j:j+4096])).numpy())
    o = np.concatenate(out)
    return o[:, :2]*ys+ym, 1/(1+np.exp(-o[:, 2:]))

if __name__ == "__main__":
    DATA, OUT, which = sys.argv[1], sys.argv[2], sys.argv[3]
    seeds = [int(s) for s in sys.argv[4].split(",")] if len(sys.argv) > 4 else [42]
    S = cargar_todo(DATA); rangos_fisicos(S); U = umbrales(S); C = clima_doy(S, MODELADAS)
    X, Y, E, ST, DT, SP, st = construir(S, U, C)
    print(X.shape, np.bincount(SP))
    ym = np.nanmean(Y[SP==0], 0); ys = np.nanstd(Y[SP==0], 0)
    for seed in seeds:
        log = []; t0 = time.time()
        fn = (lambda: LSTMNet()) if which == "lstm" else (lambda: TFT())
        m = entrenar(fn, X, Y, E, ST, SP, ym, ys, seed=seed, log=log)
        reg, pe = predecir(m, X, ST, ym, ys)
        df = pd.DataFrame({"fecha": DT, "estacion_id": ST, "split": SP, "y_PT": Y[:, 0], "y_TM": Y[:, 1],
                           "e_lluvia": E[:, 0], "e_frio": E[:, 1], f"{which}_y_PT": reg[:, 0], f"{which}_y_TM": reg[:, 1],
                           f"{which}_p_e_lluvia": pe[:, 0], f"{which}_p_e_frio": pe[:, 1]})
        df["nombre"] = [MODELADAS[i] for i in ST]
        df.to_pickle(f"{OUT}/preds_{which}_s{seed}.pkl")
        info = dict(seed=seed, tiempo_s=time.time()-t0, params=sum(p.numel() for p in m.parameters()), log=log, torch=torch.__version__)
        json.dump(info, open(f"{OUT}/info_{which}_s{seed}.json", "w"), indent=1)
        if which == "tft":
            vi = np.where(SP==2)[0][:5000]
            with torch.no_grad(): m(torch.tensor(X[vi]), torch.tensor(ST[vi]))
            w = m.last_weights.mean((0, 1)).numpy()
            pd.Series(w, index=["PT","TM","HR","sin_doy","cos_doy","faltantes"]).to_csv(f"{OUT}/tft_pesos_variables_s{seed}.csv")


In [ ]:
%%writefile evaluate.py
import sys, glob, json, pickle, numpy as np, pandas as pd
from common import *
OUT = sys.argv[1]
U = pickle.load(open(f"{OUT}/umbrales.pkl","rb"))["U"]
B = pd.read_pickle(f"{OUT}/base.pkl").reset_index().rename(columns={"index":"fecha"})
X = pd.read_pickle(f"{OUT}/preds_xgb.pkl").reset_index().rename(columns={"index":"fecha"})
X = X[["fecha","nombre","xgb_y_PT","xgb_y_TM","xgb_p_e_lluvia","xgb_p_e_frio"]]
D = B.merge(X, on=["fecha","nombre"])
# umbral por fila
D["thr_ll"] = D.nombre.map(lambda k: U[k]["p95_lluvia"]); D["thr_fr"] = D.nombre.map(lambda k: U[k]["p5_tm"])
dl = {}
for w in ["lstm","tft"]:
    fs = sorted(glob.glob(f"{OUT}/preds_{w}_s*.pkl")); dl[w] = fs
    for j, f in enumerate(fs):
        p = pd.read_pickle(f)[["fecha","nombre",f"{w}_y_PT",f"{w}_y_TM",f"{w}_p_e_lluvia",f"{w}_p_e_frio"]]
        p.columns = ["fecha","nombre"]+[f"{c}_{j}" for c in p.columns[2:]]
        D = D.merge(p, on=["fecha","nombre"], how="inner")
print("filas", len(D), D.split.value_counts().to_dict())
def split(s): return D[D.split==s]
def eval_reg(col, tgt):
    return {sp: metricas_reg(split(sp)[tgt].values, split(sp)[col].values) for sp in [1,2]}
rows = []; ev_rows = []
def add_reg(modelo, cols_pt, cols_tm):
    for var, cols, tgt in [("PT", cols_pt, "y_PT"), ("TM", cols_tm, "y_TM")]:
        ms = [metricas_reg(split(2)[tgt].values, split(2)[c].values) for c in cols]
        r = dict(Modelo=modelo, Variable=var)
        for k in ["RMSE","MAE","R2"]:
            v = np.array([m[k] for m in ms]); r[k] = v.mean(); r[k+"_sd"] = v.std() if len(v) > 1 else np.nan
        r["n"] = ms[0]["n"]; r["semillas"] = len(cols); rows.append(r)
D["clim_y_PT"] = D.clima_PT; D["clim_y_TM"] = D.clima_TM
D["pers_y_PT"] = D.PT_l0; D["pers_y_TM"] = D.TM_l0
add_reg("Climatología (línea base)", ["clim_y_PT"], ["clim_y_TM"])
add_reg("Persistencia (línea base)", ["pers_y_PT"], ["pers_y_TM"])
add_reg("XGBoost", ["xgb_y_PT"], ["xgb_y_TM"])
for w, nm in [("lstm","LSTM"),("tft","TFT")]:
    n = len(dl[w])
    if n: add_reg(nm, [f"{w}_y_PT_{j}" for j in range(n)], [f"{w}_y_TM_{j}" for j in range(n)])
REG = pd.DataFrame(rows)
# eventos
def add_ev(modelo, ev, scols, sign=1, tune=True):
    ys = [] ; res = []
    for sc in scols:
        va, te = split(1), split(2)
        sv = sign*va[sc].values; st_ = sign*te[sc].values
        t = mejor_umbral(va[ev].values, sv) if tune else 0.5
        m = ~(np.isnan(te[ev].values) | np.isnan(st_))
        r = prf(te[ev].values[m], st_[m] >= t); r["AUC_PR"] = auc_pr(te[ev].values[m], st_[m]); res.append(r)
    out = dict(Modelo=modelo, Evento={"e_lluvia":"Lluvia intensa (≥P95)","e_frio":"Día frío extremo (≤P5)"}[ev],
               prevalencia=float(split(2)[ev].mean()), semillas=len(scols))
    for k in ["Precision","Recall","F1","AUC_PR"]:
        v = np.array([r[k] for r in res]); out[k] = v.mean(); out[k+"_sd"] = v.std() if len(v) > 1 else np.nan
    for k in ["TP","FP","FN"]: out[k] = float(np.mean([r[k] for r in res]))
    ev_rows.append(out)
# persistencia de evento: valor de hoy supera umbral
D["pers_ll"] = (D.PT_l0 >= D.thr_ll).astype(float); D["pers_fr"] = (D.TM_l0 <= D.thr_fr).astype(float)
D["xgbr_ll"] = D.xgb_y_PT; D["xgbr_fr"] = -D.xgb_y_TM
add_ev("Persistencia (línea base)", "e_lluvia", ["pers_ll"], tune=False)
add_ev("Persistencia (línea base)", "e_frio", ["pers_fr"], tune=False)
add_ev("XGBoost (regresión + umbral)", "e_lluvia", ["xgbr_ll"]); add_ev("XGBoost (regresión + umbral)", "e_frio", ["xgbr_fr"])
add_ev("XGBoost (clasificador)", "e_lluvia", ["xgb_p_e_lluvia"]); add_ev("XGBoost (clasificador)", "e_frio", ["xgb_p_e_frio"])
for w, nm in [("lstm","LSTM"),("tft","TFT")]:
    n = len(dl[w])
    if n:
        add_ev(nm, "e_lluvia", [f"{w}_p_e_lluvia_{j}" for j in range(n)]); add_ev(nm, "e_frio", [f"{w}_p_e_frio_{j}" for j in range(n)])
EV = pd.DataFrame(ev_rows)
REG.to_csv(f"{OUT}/resultados_regresion.csv", index=False); EV.to_csv(f"{OUT}/resultados_eventos.csv", index=False)
# por estación (XGB clasificador y mejor DL) y por estación regresión XGB
por = []
for k in MODELADAS:
    t = D[(D.split==2)&(D.nombre==k)]
    r = dict(Estacion=k)
    for var, c, tg in [("PT","xgb_y_PT","y_PT"),("TM","xgb_y_TM","y_TM")]:
        m = metricas_reg(t[tg].values, t[c].values); r[f"RMSE_{var}"] = m["RMSE"]; r[f"R2_{var}"] = m["R2"]
    por.append(r)
pd.DataFrame(por).to_csv(f"{OUT}/resultados_por_estacion_xgb.csv", index=False)
D.to_pickle(f"{OUT}/D_eval.pkl")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(REG.round(3).to_string()); print(EV.round(3).drop(columns=["TP","FP","FN"]).to_string()); print(pd.DataFrame(por).round(3))


In [ ]:
%%writefile figures.py
import sys, pickle, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from common import *
DATA, OUT = sys.argv[1], sys.argv[2]
plt.rcParams.update({"font.family":"DejaVu Serif","font.size":9,"axes.spines.top":False,"axes.spines.right":False,"figure.dpi":150})
S = cargar_todo(DATA); rangos_fisicos(S)
# F1: disponibilidad anual PT/TM/HR
fig, axs = plt.subplots(1, 3, figsize=(10, 3.2), sharey=True)
for ax, v in zip(axs, ["PT","TM","HR"]):
    M = pd.DataFrame({k: S[k][v].notna().groupby(S[k].index.year).mean()*100 for k in ESTACIONES}).T
    im = ax.imshow(M.values, aspect="auto", cmap="Blues", vmin=0, vmax=100, extent=[1963.5, 2026.5, len(M)-0.5, -0.5])
    ax.set_title({"PT":"Precipitación (PT101)","TM":"Temperatura (TM101)","HR":"Humedad rel. (HR101)"}[v], fontsize=9)
    ax.set_xlabel("Año")
axs[0].set_yticks(range(len(ESTACIONES))); axs[0].set_yticklabels([k.title() for k in ESTACIONES])
cb = fig.colorbar(im, ax=axs, shrink=0.8, pad=0.02); cb.set_label("% de días con dato")
fig.savefig(f"{OUT}/fig_cobertura.png", bbox_inches="tight"); plt.close()
# F2: ciclo anual Puno
d = S["PUNO"]; m = d.groupby(d.index.month).agg(PT=("PT", lambda x: x.mean()*30.4), TM=("TM","mean"), HR=("HR","mean"))
fig, ax = plt.subplots(figsize=(5.5, 3)); ax.bar(range(1,13), m.PT, color="#9ec5e8", label="Precipitación media mensual (mm)")
ax.set_xticks(range(1,13)); ax.set_xticklabels(list("ENEFMAMJJASOND"[0:0]) or ["E","F","M","A","M","J","J","A","S","O","N","D"][0:12]) if False else ax.set_xticklabels(["E","F","M","A","M","J","J","A","S","O","N","D"])
ax.set_ylabel("mm/mes"); ax2 = ax.twinx(); ax2.plot(range(1,13), m.TM, "o-", color="#c0392b", label="Temp. media (°C)"); ax2.set_ylabel("°C"); ax2.spines["top"].set_visible(False)
ax.set_title("Régimen climático medio, estación Puno (1964–2026)", fontsize=9)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1+h2, l1+l2, fontsize=7, loc="upper center")
fig.savefig(f"{OUT}/fig_ciclo_puno.png", bbox_inches="tight"); plt.close()
# F3: observado vs predicho TM Puno (test, 2 años)
D = pd.read_pickle(f"{OUT}/D_eval.pkl"); p = D[(D.nombre=="PUNO")&(D.split==2)].sort_values("fecha")
p = p[(p.fecha >= "2024-01-01") & (p.fecha <= "2025-12-31")]
fig, axs = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axs[0].plot(p.fecha, p.y_TM, color="k", lw=0.8, label="Observado"); axs[0].plot(p.fecha, p.xgb_y_TM, color="#d35400", lw=0.8, label="XGBoost (t+1)")
axs[0].set_ylabel("Temp. media (°C)"); axs[0].legend(fontsize=7, ncol=2); axs[0].set_title("Estación Puno, periodo de prueba (2024–2025)", fontsize=9)
axs[1].bar(p.fecha, p.y_PT, color="#2980b9", width=1, label="Observado"); axs[1].plot(p.fecha, p.xgb_y_PT, color="#d35400", lw=0.7, label="XGBoost (t+1)")
axs[1].axhline(pickle.load(open(f"{OUT}/umbrales.pkl","rb"))["U"]["PUNO"]["p95_lluvia"], ls="--", color="gray", lw=0.8); axs[1].set_ylabel("Precip. (mm)"); axs[1].legend(fontsize=7, ncol=2)
fig.savefig(f"{OUT}/fig_obs_pred_puno.png", bbox_inches="tight"); plt.close()
# F4: importancia XGB
fig, axs = plt.subplots(1, 2, figsize=(9, 3.4))
for ax, t, ti in zip(axs, ["y_PT","y_TM"], ["Precipitación (t+1)","Temperatura (t+1)"]):
    imp = pd.read_csv(f"{OUT}/xgb_importancia_{t}.csv", index_col=0).iloc[:10, 0][::-1]
    ax.barh(imp.index, imp.values, color="#34495e"); ax.set_title(ti, fontsize=9); ax.set_xlabel("Importancia (ganancia relativa)")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_importancia.png", bbox_inches="tight"); plt.close()
# F5: F1 por modelo y evento
EV = pd.read_csv(f"{OUT}/resultados_eventos.csv"); ev = EV.Evento.unique()
fig, axs = plt.subplots(1, 2, figsize=(9, 3.3), sharey=True)
for ax, e in zip(axs, ev):
    t = EV[EV.Evento==e]; ax.barh(t.Modelo, t.F1, xerr=t.F1_sd.fillna(0), color="#16a085"); ax.set_title(e, fontsize=9); ax.set_xlabel("F1 (periodo de prueba)"); ax.invert_yaxis()
fig.tight_layout(); fig.savefig(f"{OUT}/fig_f1.png", bbox_inches="tight"); plt.close()
# F6: pesos de selección de variables TFT
import glob
W = pd.concat([pd.read_csv(f, index_col=0).iloc[:, 0] for f in sorted(glob.glob(f"{OUT}/tft_pesos_variables_s*.csv"))], axis=1).mean(1)
fig, ax = plt.subplots(figsize=(5, 2.8)); ax.bar(W.index, W.values, color="#8e44ad"); ax.set_ylabel("Peso medio de selección"); ax.set_title("TFT: importancia de variables de entrada", fontsize=9)
fig.savefig(f"{OUT}/fig_tft_pesos.png", bbox_inches="tight"); plt.close()
print("ok")


In [ ]:
%%writefile export_powerbi.py
"""Exporta CSV para Power BI: observado, predicho y nivel de alerta por estación/día (periodo de prueba)."""
import sys, pickle, numpy as np, pandas as pd
from common import *
OUT = sys.argv[1]
D = pd.read_pickle(f"{OUT}/D_eval.pkl"); U = pickle.load(open(f"{OUT}/umbrales.pkl","rb"))["U"]
def prob_cols(w, ev):
    return [c for c in D.columns if c.startswith(f"{w}_p_{ev}_")]
out = D[["fecha","nombre","split","y_PT","y_TM","e_lluvia","e_frio","xgb_y_PT","xgb_y_TM"]].copy()
sel = {}
for ev in ["e_lluvia","e_frio"]:
    cand = {"XGBoost": D[f"xgb_p_{ev}"]}
    for w, nm in [("lstm","LSTM"),("tft","TFT")]:
        cs = prob_cols(w, ev)
        if cs: cand[nm] = D[cs].mean(1)
    va = D.split == 1
    best = max(cand, key=lambda k: auc_pr(D.loc[va, ev].values, cand[k][va].values))   # selección con validación
    p = cand[best]; t = mejor_umbral(D.loc[va, ev].values, p[va].values)
    out[f"prob_{ev}"] = p; out[f"nivel_{ev}"] = np.where(p >= t, "Alerta", np.where(p >= 0.5*t, "Vigilancia", "Normal"))
    sel[ev] = dict(modelo=best, umbral=float(t))
out["fecha_objetivo"] = out.fecha + pd.Timedelta(days=1)
out = out.rename(columns={"nombre":"estacion","y_PT":"PT_obs_t+1","y_TM":"TM_obs_t+1","xgb_y_PT":"PT_pred_xgb","xgb_y_TM":"TM_pred_xgb",
                          "e_lluvia":"evento_lluvia_obs","e_frio":"evento_frio_obs","prob_e_lluvia":"prob_lluvia_intensa","prob_e_frio":"prob_frio_extremo",
                          "nivel_e_lluvia":"nivel_lluvia","nivel_e_frio":"nivel_frio"})
out["umbral_lluvia_mm"] = out.estacion.map(lambda k: U[k]["p95_lluvia"]); out["umbral_frio_C"] = out.estacion.map(lambda k: U[k]["p5_tm"])
out["periodo"] = out.split.map({0:"entrenamiento",1:"validacion",2:"prueba"}); out = out.drop(columns="split")
out.to_csv(f"{OUT}/powerbi_predicciones_alertas.csv", index=False, encoding="utf-8-sig")
import json; json.dump(sel, open(f"{OUT}/powerbi_modelo_seleccionado.json","w"), indent=1)
# resumen de alertas en prueba
t = out[out.periodo=="prueba"]
for ev, c in [("lluvia","evento_lluvia_obs"),("frio","evento_frio_obs")]:
    g = t.groupby(f"nivel_{ev}").agg(dias=(c,"size"), eventos=(c,"sum")); g["tasa_evento_%"] = 100*g.eventos/g.dias; print(ev, sel["e_lluvia" if ev=="lluvia" else "e_frio"]); print(g.round(2))
    g.to_csv(f"{OUT}/alertas_resumen_{ev}.csv")


In [ ]:
!pip -q install xgboost

In [ ]:
!python run_xgb.py $DATA $OUT

In [ ]:
!python dl.py $DATA $OUT lstm 42,43,44,45,46

In [ ]:
# ~15 min por semilla en CPU; use GPU (Runtime > Change runtime type) para acelerar
!python dl.py $DATA $OUT tft 42,43,44

In [ ]:
!python evaluate.py $OUT

In [ ]:
!python figures.py $DATA $OUT

In [ ]:
!python export_powerbi.py $OUT

Los CSV de `OUT` (`powerbi_predicciones_alertas.csv`, `resultados_*.csv`) alimentan el tablero Power BI.